# Batch 1 — Pre-Scan (Week 0 DEXA)



In [ ]:
import pandas as pd
from pathlib import Path
csv_path = Path(r'../data/batch1_pre_scan_summary.csv').resolve()
try:
    df = pd.read_csv(csv_path)
    df
except Exception:
    print('Failed to load', csv_path)


## Batch 1 — 1-week post-treatment summary


In [1]:
# Load week-1 master CSV if present, otherwise scan for week-1 TXT files and build one; then display rows for this batch
from pathlib import Path
import re

DATA_CSV = Path(r'../data/week1_reports.csv').resolve()
batch_name = 'Batch 1'

def scan_and_build(downloads_root):
    week1_re = re.compile(r'(?:week[\s_-]*1|1[\s_-]*week|wk[\s_-]*1|week1|1week)', re.I)
    patterns = {
        'sample_area': re.compile(r"Sample Area:\s*([0-9.]+)\s*cm\^2"),
        'bone_area': re.compile(r"Bone Area:\s*([0-9.]+)\s*cm\^2"),
        'total_weight': re.compile(r"Total Weight:\s*([0-9.]+)\s*g"),
        'soft_weight': re.compile(r"Soft Weight:\s*([0-9.]+)\s*g"),
        'lean_weight': re.compile(r"Lean Weight:\s*([0-9.]+)\s*g"),
        'fat_weight': re.compile(r"Fat Weight:\s*([0-9.]+)\s*g"),
        'fat_percent': re.compile(r"Fat Percent:\s*([0-9.]+)"),
        'BMC': re.compile(r"BMC:\s*([0-9.]+)\s*g"),
        'BMD': re.compile(r"BMD:\s*([0-9.]+)\s*mg/cm\^2"),
    }
    rows = []
    for txt in downloads_root.rglob('*.txt'):
        nl = str(txt).lower()
        if week1_re.search(nl):
            text = txt.read_text(encoding='utf-8', errors='replace')
            inside_block = ''
            whole_block = ''
            if 'INSIDE ROI TISSUE STATISTICS:' in text and 'WHOLE TISSUE STATISTICS:' in text:
                inside_block = text.split('INSIDE ROI TISSUE STATISTICS:')[1].split('WHOLE TISSUE STATISTICS:')[0]
                whole_block = text.split('WHOLE TISSUE STATISTICS:')[1]
            else:
                inside_block = text
            parts = txt.parts
            batch = next((p for p in parts if p.lower().startswith('batch')), '')
            sex = 'Male' if any(p.lower()=='male' for p in parts) else ('Female' if any(p.lower()=='female' for p in parts) else '')
            row = {'batch': batch, 'sex': sex, 'filename': txt.name}
            for k,p in patterns.items():
                mi = p.search(inside_block)
                mw = p.search(whole_block)
                row[f'inside_{k}'] = mi.group(1) if mi else ''
                row[f'whole_{k}'] = mw.group(1) if mw else ''
            rows.append(row)
    return rows

try:
    import pandas as pd
    if DATA_CSV.exists():
        master = pd.read_csv(DATA_CSV)
    else:
        downloads_root = Path(r"c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA Scans")
        rows = scan_and_build(downloads_root)
        master = pd.DataFrame(rows)
        try:
            master.to_csv(DATA_CSV, index=False)
        except Exception:
            pass
    if not master.empty:
        df_batch = master[master['batch'].str.lower()==batch_name.lower()]
        if not df_batch.empty:
            display(df_batch.reset_index(drop=True))
        else:
            print('No week-1 rows for', batch_name)
    else:
        print('No week-1 files found anywhere')
except Exception as e:
    print('Error building/displaying week-1 table:', e)


,batch,sex,filename,path,inside_sample_area,whole_sample_area,inside_bone_area,whole_bone_area,inside_total_weight,whole_total_weight,...,inside_lean_weight,whole_lean_weight,inside_fat_weight,whole_fat_weight,inside_fat_percent,whole_fat_percent,inside_BMC,whole_BMC,inside_BMD,whole_BMD
0,Batch 1,NaN,B1_M_2 Week 1 Redo.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,23.071,33.309,10.238,12.859,23.7617,34.4093,...,13.8447,19.5697,9.0695,13.7281,39.580,41.228,0.84753,1.11155,82.785,86.440
1,Batch 1,Female,B1_F_1 Week 4.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,28.076,29.158,9.149,9.579,25.3930,26.4028,...,17.9436,18.5506,6.7569,7.1136,27.355,27.718,0.69242,0.73863,75.679,77.110
2,Batch 1,Male,B1_M_1 Week 4.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,28.670,28.834,8.810,8.913,29.2905,29.5263,...,21.0112,21.1039,7.6123,7.7423,26.595,26.840,0.66695,0.68010,75.702,76.301
3,Batch 1,Female,B1_F_1 Week 2.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,27.181,27.436,8.278,8.407,25.9537,26.2504,...,18.2934,18.4088,7.0727,7.2384,27.883,28.223,0.58754,0.60324,70.977,71.759
4,Batch 1,Male,B1_M_1 Week 2.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,28.123,29.438,8.464,8.777,30.0309,31.3001,...,21.1144,21.8478,8.2732,8.7813,28.152,28.670,0.64339,0.67104,76.011,76.454
5,Batch 1,Female,B1_F_0 Week 1.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,29.742,32.851,8.803,10.463,24.8310,27.9316,...,16.8183,18.5689,7.4227,8.5619,30.621,31.558,0.58998,0.80076,67.019,76.534
6,Batch 1,Female,B1_F_1 Week 1.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,30.906,34.226,8.340,9.981,26.0637,29.6267,...,19.6600,21.8363,5.8182,6.9775,22.836,24.216,0.58548,0.81286,70.198,81.444
7,Batch 1,Female,B1_F_2 Week 1.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,30.350,33.502,8.898,10.722,29.6166,33.2939,...,18.0453,19.1080,10.9470,13.2603,37.758,40.967,0.62434,0.92561,70.166,86.328
8,Batch 1,Female,B1_F_3 Week 1.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,30.132,32.405,8.861,10.026,28.4473,30.8878,...,19.0449,20.1605,8.7794,9.9397,31.553,33.022,0.62297,0.78761,70.303,78.559
9,Batch 1,Female,B1_F_4 Week 1.txt,c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA S...,31.222,33.738,8.990,9.981,28.1668,30.5251,...,18.9901,20.4675,8.5375,9.3197,31.014,31.287,0.63919,0.73790,71.102,73.931


## Batch 1 — 2-week post-treatment summary



In [2]:
# Batch 1 — 2-week post-treatment summary
# Scan/build week-2 master CSV and display rows for this batch
from pathlib import Path
import re

DATA_CSV2 = Path(r'../data/week2_reports.csv').resolve()
batch_name = 'Batch 1'

def scan_and_build_week(downloads_root, week_num):
    week_re = re.compile(rf'(?:week[\s_-]*{week_num}|{week_num}[\s_-]*week|wk[\s_-]*{week_num}|week{week_num}|{week_num}week)', re.I)
    patterns = {
        'sample_area': re.compile(r"Sample Area:\s*([0-9.]+)\s*cm\^2"),
        'bone_area': re.compile(r"Bone Area:\s*([0-9.]+)\s*cm\^2"),
        'total_weight': re.compile(r"Total Weight:\s*([0-9.]+)\s*g"),
        'soft_weight': re.compile(r"Soft Weight:\s*([0-9.]+)\s*g"),
        'lean_weight': re.compile(r"Lean Weight:\s*([0-9.]+)\s*g"),
        'fat_weight': re.compile(r"Fat Weight:\s*([0-9.]+)\s*g"),
        'fat_percent': re.compile(r"Fat Percent:\s*([0-9.]+)"),
        'BMC': re.compile(r"BMC:\s*([0-9.]+)\s*g"),
        'BMD': re.compile(r"BMD:\s*([0-9.]+)\s*mg/cm\^2"),
    }
    rows = []
    for txt in downloads_root.rglob('*.txt'):
        nl = str(txt).lower()
        if week_re.search(nl):
            text = txt.read_text(encoding='utf-8', errors='replace')
            inside_block = ''
            whole_block = ''
            if 'INSIDE ROI TISSUE STATISTICS:' in text and 'WHOLE TISSUE STATISTICS:' in text:
                inside_block = text.split('INSIDE ROI TISSUE STATISTICS:')[1].split('WHOLE TISSUE STATISTICS:')[0]
                whole_block = text.split('WHOLE TISSUE STATISTICS:')[1]
            else:
                inside_block = text
            parts = txt.parts
            batch = next((p for p in parts if p.lower().startswith('batch')), '')
            sex = 'Male' if any(p.lower()=='male' for p in parts) else ('Female' if any(p.lower()=='female' for p in parts) else '')
            row = {'batch': batch, 'sex': sex, 'filename': txt.name}
            for k,p in patterns.items():
                mi = p.search(inside_block)
                mw = p.search(whole_block)
                row[f'inside_{k}'] = mi.group(1) if mi else ''
                row[f'whole_{k}'] = mw.group(1) if mw else ''
            rows.append(row)
    return rows

# Build or load week-2 master and display
try:
    import pandas as pd
    if DATA_CSV2.exists():
        master2 = pd.read_csv(DATA_CSV2)
    else:
        downloads_root = Path(r"c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA Scans")
        rows2 = scan_and_build_week(downloads_root, 2)
        master2 = pd.DataFrame(rows2)
        try:
            master2.to_csv(DATA_CSV2, index=False)
        except Exception:
            pass
    if not master2.empty:
        df_batch2 = master2[master2['batch'].str.lower()==batch_name.lower()]
        if not df_batch2.empty:
            display(df_batch2.reset_index(drop=True))
        else:
            print('No week-2 rows for', batch_name)
    else:
        print('No week-2 files found anywhere')
except Exception as e:
    print('Error building/displaying week-2 table:', e)


,batch,sex,filename,inside_sample_area,whole_sample_area,inside_bone_area,whole_bone_area,inside_total_weight,whole_total_weight,inside_soft_weight,...,inside_lean_weight,whole_lean_weight,inside_fat_weight,whole_fat_weight,inside_fat_percent,whole_fat_percent,inside_BMC,whole_BMC,inside_BMD,whole_BMD
0,Batch 1,,B1_M_2 Week 1 Redo.txt,23.071,33.309,10.238,12.859,23.7617,34.4093,22.9142,...,13.8447,19.5697,9.0695,13.7281,39.580,41.228,0.84753,1.11155,82.785,86.440
1,Batch 1,Female,B1_F_2 Week 4.txt,29.027,29.871,9.714,9.868,28.6503,29.2977,27.8563,...,16.2718,16.5102,11.5845,11.9738,41.587,42.037,0.79401,0.81368,81.736,82.461
2,Batch 1,Male,B1_M_2 Week 4.txt,28.437,29.844,8.424,8.798,28.3642,29.9219,27.7527,...,21.5638,22.6645,6.1889,6.6126,22.300,22.586,0.61149,0.64476,72.588,73.287
3,Batch 1,Female,B1_F_0 Week 2.txt,27.406,28.361,8.526,8.742,24.1541,24.8962,23.5866,...,15.0724,15.3373,8.5142,8.9677,36.098,36.897,0.56753,0.59124,66.562,67.629
4,Batch 1,Female,B1_F_1 Week 2.txt,27.181,27.436,8.278,8.407,25.9537,26.2504,25.3661,...,18.2934,18.4088,7.0727,7.2384,27.883,28.223,0.58754,0.60324,70.977,71.759
5,Batch 1,Female,B1_F_2 Week 2.txt,30.217,31.141,9.653,10.007,30.8019,31.9114,30.0504,...,16.9833,17.5007,13.0671,13.6215,43.484,43.768,0.75153,0.78922,77.858,78.865
6,Batch 1,Female,B1_F_3 Week 2.txt,30.129,32.775,9.213,10.212,28.3000,30.7730,27.6354,...,17.6318,18.8508,10.0036,11.1439,36.198,37.153,0.66466,0.77825,72.147,76.210
7,Batch 1,Female,B1_F_4 Week 2.txt,27.981,28.860,9.653,9.903,27.7403,28.5915,26.9733,...,17.1762,17.4867,9.7971,10.3020,36.321,37.073,0.76708,0.80278,79.464,81.062
8,Batch 1,Male,B1_M_0 Week 2.txt,29.020,30.124,8.321,8.574,29.0777,30.2365,28.5024,...,20.9750,21.6872,7.5274,7.9511,26.410,26.827,0.57529,0.59819,69.136,69.766
9,Batch 1,Male,B1_M_1 Week 2.txt,28.123,29.438,8.464,8.777,30.0309,31.3001,29.3875,...,21.1144,21.8478,8.2732,8.7813,28.152,28.670,0.64339,0.67104,76.011,76.454


## Batch 1 — 3-week post-treatment summary



In [3]:
# Batch 1 — 3-week post-treatment summary
# Scan/build week-3 master CSV and display rows for this batch (same logic as week-2)
from pathlib import Path
import re

DATA_CSV3 = Path(r'../data/week3_reports.csv').resolve()
batch_name = 'Batch 1'

# reuse scan_and_build_week from above
try:
    import pandas as pd
    if DATA_CSV3.exists():
        master3 = pd.read_csv(DATA_CSV3)
    else:
        downloads_root = Path(r"c:\Users\hanss\Downloads\DEXA Scans (1)\DEXA Scans")
        rows3 = scan_and_build_week(downloads_root, 3)
        master3 = pd.DataFrame(rows3)
        try:
            master3.to_csv(DATA_CSV3, index=False)
        except Exception:
            pass
    if not master3.empty:
        df_batch3 = master3[master3['batch'].str.lower()==batch_name.lower()]
        if not df_batch3.empty:
            display(df_batch3.reset_index(drop=True))
        else:
            print('No week-3 rows for', batch_name)
    else:
        print('No week-3 files found anywhere')
except Exception as e:
    print('Error building/displaying week-3 table:', e)


,batch,sex,filename,inside_sample_area,whole_sample_area,inside_bone_area,whole_bone_area,inside_total_weight,whole_total_weight,inside_soft_weight,...,inside_lean_weight,whole_lean_weight,inside_fat_weight,whole_fat_weight,inside_fat_percent,whole_fat_percent,inside_BMC,whole_BMC,inside_BMD,whole_BMD
0,Batch 1,Female,B1_F_0 Week 4.txt,28.371,29.720,8.305,8.702,24.0639,25.1351,23.5121,...,15.7656,16.3674,7.7465,8.1835,32.947,33.333,0.55173,0.58417,66.436,67.132
1,Batch 1,Female,B1_F_1 Week 4.txt,28.076,29.158,9.149,9.579,25.3930,26.4028,24.7005,...,17.9436,18.5506,6.7569,7.1136,27.355,27.718,0.69242,0.73863,75.679,77.110
2,Batch 1,Female,B1_F_2 Week 4.txt,29.027,29.871,9.714,9.868,28.6503,29.2977,27.8563,...,16.2718,16.5102,11.5845,11.9738,41.587,42.037,0.79401,0.81368,81.736,82.461
3,Batch 1,Female,B1_F_3 Week 4.txt,29.657,30.581,8.684,8.948,27.3143,28.1561,26.6784,...,17.3800,17.7815,9.2984,9.7123,34.854,35.325,0.63597,0.66230,73.235,74.013
4,Batch 1,Female,B1_F_4 Week 4.txt,28.815,29.830,10.049,10.345,27.3370,28.2712,26.4965,...,17.7165,18.0782,8.7800,9.3135,33.137,34.001,0.84046,0.87952,83.638,85.015
5,Batch 1,Male,B1_M_0 Week 4.txt,29.260,29.927,8.316,8.439,27.7035,28.5474,27.1240,...,20.9648,21.5692,6.1592,6.3906,22.708,22.856,0.57948,0.58769,69.681,69.642
6,Batch 1,Male,B1_M_1 Week 4.txt,28.670,28.834,8.810,8.913,29.2905,29.5263,28.6236,...,21.0112,21.1039,7.6123,7.7423,26.595,26.840,0.66695,0.68010,75.702,76.301
7,Batch 1,Male,B1_M_2 Week 4.txt,28.437,29.844,8.424,8.798,28.3642,29.9219,27.7527,...,21.5638,22.6645,6.1889,6.6126,22.300,22.586,0.61149,0.64476,72.588,73.287
8,Batch 1,Male,B1_M_3 Week 4.txt,31.833,32.787,9.352,9.499,30.0421,30.9094,29.3857,...,22.6413,23.2217,6.7444,7.0185,22.951,23.209,0.65643,0.66916,70.188,70.445
9,Batch 1,Male,B1_M_4 Week 4.txt,29.949,31.151,9.772,10.078,31.3514,32.5611,30.5476,...,23.2205,23.9459,7.3270,7.7790,23.986,24.520,0.80381,0.83618,82.258,82.970
